# AudioSmith AI 🎙️✨ - DeepFilterNet Fine-Tuning

This notebook fine-tunes the official DeepFilterNet model using the AudioSmith AI machine learning pipeline. It is fully self-contained and orchestrates the dataset preparation, training, evaluation, and export steps automatically.

### ⚠️ Instructions
1. **Enable GPU**: Ensure the Accelerator in the right panel is set to `GPU P100` or `GPU T4x2`.
2. **Internet**: Ensure Internet is toggled **ON** in the right panel so the repository and datasets can be downloaded.
3. **Run All**: Click **Run All** to start the pipeline.

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Training will be extremely slow. Please enable a GPU in the Kaggle settings.")

## 1. Extract Local Codebase & Install Dependencies
This cell extracts the AudioSmith repository code directly from a Base64 blob. This prevents GitHub caching issues on Kaggle.

In [ ]:
import os
import base64
import zipfile
import io
import subprocess
import sys

REPO_DIR = '/kaggle/working/AudioSmith'
os.makedirs(REPO_DIR, exist_ok=True)
os.chdir(REPO_DIR)

print("Extracting repository code...")
repo_b64 = """..."""
zip_data = base64.b64decode(repo_b64)
with zipfile.ZipFile(io.BytesIO(zip_data)) as zf:
    zf.extractall('.')
os.makedirs('scripts', exist_ok=True)

print("Installing Rust (required for DeepFilterNet on Python 3.12+)...")
subprocess.run('curl --proto \'=https\' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y', shell=True, check=True)
os.environ['PATH'] += ':/root/.cargo/bin'

deps = ["torch>=2.4.0", "torchaudio>=2.4.0", "deepfilternet>=0.5.6", "numpy>=1.26.0", "librosa>=0.10.2", "soundfile>=0.12.1", "pyyaml>=6.0.1", "mlflow>=2.15.0", "torchmetrics>=1.4.0", "pesq>=0.0.4", "pystoi>=0.3.3", "requests>=2.31.0", "onnx>=1.15.0"]
print('Installing dependencies:', deps)
subprocess.run([sys.executable, '-m', 'pip', 'install'] + deps, check=True)


## 2. Prepare Datasets (Bypassing VoiceBank)
We dynamically write a highly robust `download_assets.sh` script to only download LibriSpeech and MUSAN, skipping VoiceBank entirely to avoid datashare.ed.ac.uk's IP blocking.

In [ ]:
os.chdir(REPO_DIR)
dl_script = """#!/usr/bin/env bash
set -euo pipefail
DATASET_ROOT="${DATASET_ROOT:-datasets}"
mkdir -p "${DATASET_ROOT}"

echo "Downloading LibriSpeech (train-clean-100)..."
if [ ! -d "${DATASET_ROOT}/LibriSpeech/train-clean-100" ]; then
    curl -L "http://www.openslr.org/resources/12/train-clean-100.tar.gz" | tar -xz -C "${DATASET_ROOT}"
fi

echo "Downloading MUSAN..."
if [ ! -d "${DATASET_ROOT}/musan" ]; then
    curl -L "https://www.openslr.org/resources/17/musan.tar.gz" | tar -xz -C "${DATASET_ROOT}"
fi
echo "Assets ready!"
"""
with open('scripts/download_assets.sh', 'w') as f:
    f.write(dl_script)
!chmod +x scripts/download_assets.sh
!export DATASET_ROOT=./ml/data && ./scripts/download_assets.sh


## 3. Patch Pipeline for Synthetic Validation
Since we bypassed VoiceBank, we will patch `finetune.py` and `evaluate.py` to use a subset of LibriSpeech + MUSAN for the validation loop.

In [ ]:
os.chdir(REPO_DIR)
def patch_file(filepath):
    with open(filepath, 'r') as f:
        content = f.read()
    
    # Replace ValidationDataset imports and usage with NoisyCleanDataset
    content = content.replace('ValidationDataset', 'NoisyCleanDataset')
    # Update the instantiation to use train dataset args but with seed 1337
    content = content.replace(
        "clean_dir=config['validation']['clean_dir'],\n        noisy_dir=config['validation']['noisy_dir'],",
        "clean_dir=config['dataset']['clean_dir'],\n        noise_dir=config['dataset']['noise_dir'],\n        snr_range=config['dataset']['snr_range'],\n        seed=1337,"
    )
    with open(filepath, 'w') as f:
        f.write(content)

patch_file('ml/scripts/finetune.py')
patch_file('ml/scripts/evaluate.py')
print("Scripts successfully patched for synthetic validation.")


## 4. Fine-Tune DeepFilterNet
We run the fine-tuning pipeline. DeepFilterNet's official weights are automatically downloaded internally. The trainer tracks progress via MLflow and saves checkpoints.

In [ ]:
os.chdir(REPO_DIR)
!export PYTHONPATH=. && python ml/scripts/finetune.py

## 5. Evaluate Fine-Tuned Model


In [ ]:
os.chdir(REPO_DIR)
!export PYTHONPATH=. && python ml/scripts/evaluate.py --checkpoint ml/checkpoints/best_model.pt

## 6. Export Model to ONNX


In [ ]:
os.chdir(REPO_DIR)
!mkdir -p ml/exports
!export PYTHONPATH=. && python ml/scripts/export_model.py --model deepfilternet --checkpoint ml/checkpoints/best_model.pt --output ml/exports/fine_tuned.onnx --format onnx

## 7. Package and Export Artifacts


In [ ]:
os.chdir(REPO_DIR)
!zip -r /kaggle/working/AudioSmith_Finetuned_Model.zip ml/checkpoints ml/exports ml/mlruns
print("\n=========================================================")
print("✅ SUCCESS!\n")
print("Your fine-tuned model has been packaged into `AudioSmith_Finetuned_Model.zip`.")
print("\n📥 INSTRUCTIONS FOR DEPLOYMENT:")
print("1. Look at the 'Output' panel on the right side of this Kaggle notebook.")
print("2. Download `AudioSmith_Finetuned_Model.zip`.")
print("=========================================================")